In [ ]:
%%capture
!pip install --upgrade "kaggle-environments>=1.28.0"

In [ ]:
import math
import sys
sys.path.insert(0, '/home/t/orbitwars')
from visualizer import Visualizer
from kaggle_environments import make
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet, CENTER, ROTATION_RADIUS_LIMIT

viz = Visualizer()

MAX_DISTANCE = 40
LOOK_AHEAD = 20


def build_proximity_graph(planets, obs, max_distance=MAX_DISTANCE, look_ahead=LOOK_AHEAD):
    """Build adjacency list: planet_id -> list of (neighbor_id, dist) within max_distance.

    Orbiting planets are projected LOOK_AHEAD turns into the future before
    computing distances so the graph reflects where they will be, not where
    they are now.
    """
    cx = cy = CENTER
    initial_planets = obs.get('initial_planets', [])
    angular_velocity = obs.get('angular_velocity', 0.0)
    step = obs.get('step', 0)

    # Project each planet to its future position.
    # Orbiting planets (inside ROTATION_RADIUS_LIMIT) are advanced by look_ahead
    # turns using their initial angle + accumulated angular displacement.
    # Static planets keep their current position.
    future_pos = {}
    for p in planets:
        r = math.sqrt((p.x - cx) ** 2 + (p.y - cy) ** 2)
        if r + p.radius < ROTATION_RADIUS_LIMIT:
            ip = next((Planet(*ip) for ip in initial_planets if ip[0] == p.id), None)
            if ip is not None:
                initial_angle = math.atan2(ip.y - cy, ip.x - cx)
                future_angle = initial_angle + angular_velocity * (step + look_ahead)
                future_pos[p.id] = (cx + r * math.cos(future_angle), cy + r * math.sin(future_angle))
                continue
        future_pos[p.id] = (p.x, p.y)

    # Connect every pair of planets whose projected distance is within max_distance.
    graph = {p.id: [] for p in planets}
    for i, a in enumerate(planets):
        ax, ay = future_pos[a.id]
        for b in planets[i + 1:]:
            bx, by = future_pos[b.id]
            dist = math.sqrt((ax - bx) ** 2 + (ay - by) ** 2)
            if dist <= max_distance:
                graph[a.id].append((b.id, dist))
                graph[b.id].append((a.id, dist))
    return graph, future_pos


def viz_proximity_graph(viz, step, planets, graph, future_pos):
    """Draw graph edges and future-position planet labels onto the visualizer frame."""
    seen_edges = set()
    for pid, neighbors in graph.items():
        fpx, fpy = future_pos[pid]
        viz.add_label(step, fpx, fpy, f'P{pid}', color='#22ffcc')
        for nid, dist in neighbors:
            edge = (min(pid, nid), max(pid, nid))
            if edge in seen_edges:
                continue
            seen_edges.add(edge)
            nx, ny = future_pos[nid]
            viz.add_line(step, fpx, fpy, nx, ny, color='#22aaff', width=1)


def hellburner(obs):
    moves = []
    player = obs['player']
    planets = [Planet(*p) for p in obs['planets']]

    viz.record(obs)
    step = obs['step']

    graph, future_pos = build_proximity_graph(planets, obs)
    #viz_proximity_graph(viz, step, planets, graph, future_pos)

    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not targets:
        return moves

    for mine in my_planets:
        nearest = None
        min_dist = float('inf')
        for t in targets:
            dist = math.sqrt((mine.x - t.x) ** 2 + (mine.y - t.y) ** 2)
            if dist < min_dist:
                min_dist = dist
                nearest = t

        if nearest is None:
            continue

        ships_needed = max(nearest.ships + 1, 20)

        if mine.ships >= ships_needed:
            angle = math.atan2(nearest.y - mine.y, nearest.x - mine.x)
            viz.add_line(step, mine.x, mine.y, nearest.x, nearest.y, color='#ff4444', width=1)
            viz.add_text(step, f'P{mine.id} -> P{nearest.id} ({ships_needed} ships, dist={min_dist:.1f})')
            moves.append([mine.id, angle, ships_needed])

    return moves

In [ ]:
# Run game and save visualizer
env = make('orbit_wars', debug=False)
env.run([hellburner, 'random'])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f'Player {i}: reward={s.reward}, status={s.status}')

#env.render(mode='ipython', width=800, height=600)
viz.save('/mnt/c/Users/ajohn/Downloads/orbitwars_viz.html')